# XGBoost Fraud Detection Model
Train an XGBoost classifier on `ML.PROJECTS.FRAUD_ML_TEST_ONLINE` with engineered features for real-time fraud prediction.

In [ ]:
from snowflake.snowpark.context import get_active_session
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.metrics import (
    classification_report, confusion_matrix, roc_auc_score,
    precision_recall_curve, auc, f1_score, precision_score, recall_score
)
from sklearn.preprocessing import LabelEncoder
import xgboost as xgb
import matplotlib.pyplot as plt
import seaborn as sns

session = get_active_session()
print('Session ready')

In [ ]:
%%sql -r raw_data
SELECT * FROM ML.PROJECTS.FRAUD_ML_TEST_ONLINE

In [ ]:
df = raw_data.copy()
print(f"Shape: {df.shape}")
print(f"Fraud rate: {df['IS_FRAUD'].mean():.4f}")
print(df.dtypes)
df.head()

In [ ]:
df['TRANSACTION_TIME'] = pd.to_datetime(df['TRANSACTION_TIME'])

df['LOG_AMOUNT'] = np.log1p(df['AMOUNT'])

bins = [0, 50, 200, 500, 1000, 2500, 5000, float('inf')]
labels = [0, 1, 2, 3, 4, 5, 6]
df['AMOUNT_BUCKET'] = pd.cut(df['AMOUNT'], bins=bins, labels=labels).astype(int)

df['HOUR_OF_DAY'] = df['TRANSACTION_TIME'].dt.hour
df['IS_NIGHT'] = ((df['HOUR_OF_DAY'] < 7) | (df['HOUR_OF_DAY'] > 22)).astype(int)
df['DAY_OF_WEEK'] = df['TRANSACTION_TIME'].dt.dayofweek
df['IS_WEEKEND'] = (df['DAY_OF_WEEK'] >= 5).astype(int)

unusual_locations = ['Lagos', 'Minsk', 'Pyongyang', 'Caracas', 'Mogadishu', 'Tbilisi', 'Chisinau', 'Tirana']
df['IS_UNUSUAL_LOCATION'] = df['LOCATION'].isin(unusual_locations).astype(int)

risky_merchants = ['QuickCash ATM', 'CryptoXchange', 'GoldBullion Direct', 'WireTransfer Global']
df['IS_RISKY_MERCHANT'] = (df['MERCHANT'].isin(risky_merchants) | df['MERCHANT'].str.startswith('UnknownVendor_')).astype(int)

def merchant_category(m):
    if m in risky_merchants or m.startswith('UnknownVendor_'):
        return 'risky'
    elif m in ['Amazon', 'Walmart', 'Target', 'Costco', 'Home Depot', 'Best Buy']:
        return 'retail'
    elif m in ['Starbucks', 'McDonalds', 'Chipotle', 'DoorDash', 'Trader Joes', 'Whole Foods']:
        return 'food'
    elif m in ['Netflix', 'Spotify', 'Uber', 'Lyft']:
        return 'digital'
    elif m in ['Shell Gas', 'Chevron']:
        return 'gas'
    elif m in ['CVS Pharmacy', 'Walgreens']:
        return 'pharmacy'
    elif m in ['ElectroMart', 'JewelryPalace']:
        return 'luxury'
    else:
        return 'other'

df['MERCHANT_CATEGORY'] = df['MERCHANT'].apply(merchant_category)

le_merchant_cat = LabelEncoder()
df['MERCHANT_CATEGORY_ENC'] = le_merchant_cat.fit_transform(df['MERCHANT_CATEGORY'])

le_location = LabelEncoder()
df['LOCATION_ENC'] = le_location.fit_transform(df['LOCATION'])

df['IS_HIGH_AMOUNT'] = (df['AMOUNT'] > 1000).astype(int)
df['RISK_FLAG_COUNT'] = df['IS_NIGHT'] + df['IS_UNUSUAL_LOCATION'] + df['IS_RISKY_MERCHANT'] + df['IS_HIGH_AMOUNT']

print(f"Features engineered. Shape: {df.shape}")
print(f"\nRisk flag distribution:")
print(df.groupby('RISK_FLAG_COUNT')['IS_FRAUD'].agg(['count', 'mean']).round(4))

In [ ]:
FEATURES = [
    'AMOUNT', 'LOG_AMOUNT', 'AMOUNT_BUCKET',
    'HOUR_OF_DAY', 'IS_NIGHT', 'DAY_OF_WEEK', 'IS_WEEKEND',
    'IS_UNUSUAL_LOCATION', 'LOCATION_ENC',
    'IS_RISKY_MERCHANT', 'MERCHANT_CATEGORY_ENC',
    'IS_HIGH_AMOUNT', 'RISK_FLAG_COUNT'
]
TARGET = 'IS_FRAUD'

X = df[FEATURES].copy()
y = df[TARGET].astype(int).copy()

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

fraud_ratio = (y_train == 0).sum() / max((y_train == 1).sum(), 1)
print(f"Train: {X_train.shape[0]} rows, Test: {X_test.shape[0]} rows")
print(f"Train fraud rate: {y_train.mean():.4f}")
print(f"scale_pos_weight: {fraud_ratio:.1f}")

In [ ]:
model = xgb.XGBClassifier(
    n_estimators=300,
    max_depth=5,
    learning_rate=0.05,
    scale_pos_weight=fraud_ratio,
    subsample=0.8,
    colsample_bytree=0.8,
    min_child_weight=3,
    gamma=1,
    reg_alpha=0.1,
    reg_lambda=1.0,
    eval_metric='aucpr',
    random_state=42,
    use_label_encoder=False
)

model.fit(
    X_train, y_train,
    eval_set=[(X_test, y_test)],
    verbose=50
)

print('Training complete.')

In [ ]:
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

roc_auc = roc_auc_score(y_test, y_proba)
precision_vals, recall_vals, _ = precision_recall_curve(y_test, y_proba)
pr_auc = auc(recall_vals, precision_vals)

print('=== Model Evaluation ===')
print(f'ROC-AUC:  {roc_auc:.4f}')
print(f'PR-AUC:   {pr_auc:.4f}')
print(f'F1:       {f1_score(y_test, y_pred):.4f}')
print(f'Precision:{precision_score(y_test, y_pred):.4f}')
print(f'Recall:   {recall_score(y_test, y_pred):.4f}')
print()
print(classification_report(y_test, y_pred, target_names=['Legit', 'Fraud']))
print('Confusion Matrix:')
print(confusion_matrix(y_test, y_pred))

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

feat_imp = pd.Series(model.feature_importances_, index=FEATURES).sort_values(ascending=True)
feat_imp.plot(kind='barh', ax=axes[0], color='steelblue')
axes[0].set_title('Feature Importance (Gain)')
axes[0].set_xlabel('Importance')

axes[1].plot(recall_vals, precision_vals, color='darkorange', lw=2)
axes[1].set_title(f'Precision-Recall Curve (AUC={pr_auc:.3f})')
axes[1].set_xlabel('Recall')
axes[1].set_ylabel('Precision')
axes[1].set_xlim([0, 1])
axes[1].set_ylim([0, 1.05])
axes[1].axhline(y=y_test.mean(), color='gray', linestyle='--', label='Baseline')
axes[1].legend()

plt.tight_layout()
plt.show()

In [ ]:
fig, ax = plt.subplots(figsize=(6, 5))
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
            xticklabels=['Legit', 'Fraud'],
            yticklabels=['Legit', 'Fraud'], ax=ax)
ax.set_xlabel('Predicted')
ax.set_ylabel('Actual')
ax.set_title('Confusion Matrix')
plt.tight_layout()
plt.show()

In [ ]:
from snowflake.ml.registry import Registry
from snowflake.ml.model.model_signature import FeatureSpec, DataType, ModelSignature
from snowflake.ml.model import task

reg = Registry(session=session, database_name='ML', schema_name='PROJECTS')

input_features = [FeatureSpec(name=f, dtype=DataType.DOUBLE) for f in FEATURES]
output_features = [FeatureSpec(name='IS_FRAUD', dtype=DataType.INT64)]
sig = ModelSignature(inputs=input_features, outputs=output_features)

mv = reg.log_model(
    model=model,
    model_name='FRAUD_DETECTION_XGBOOST',
    version_name='V1',
    signatures={'predict': sig},
    target_platforms=['SNOWPARK_CONTAINER_SERVICES'],
    metrics={
        'roc_auc': float(roc_auc),
        'pr_auc': float(pr_auc),
        'f1': float(f1_score(y_test, y_pred)),
        'precision': float(precision_score(y_test, y_pred)),
        'recall': float(recall_score(y_test, y_pred))
    },
    task=task.Task.TABULAR_BINARY_CLASSIFICATION,
    comment='XGBoost fraud detector trained on ML.PROJECTS.FRAUD_ML_TEST_ONLINE with 12 engineered features',
    sample_input_data=X_test.head(10)
)

print(f'Model registered: ML.PROJECTS.FRAUD_DETECTION_XGBOOST version V1')
print(f'Target platform: SNOWPARK_CONTAINER_SERVICES')
print(f'Metrics: ROC-AUC={roc_auc:.4f}, PR-AUC={pr_auc:.4f}')

In [ ]:
print('Model logged with target_platform=SNOWPARK_CONTAINER_SERVICES')
print('Warehouse inference is disabled. Will test via SPCS service after deployment.')
print(f'\nModel version: {mv.model_name} / {mv.version_name}')
print(f'Functions: {mv.show_functions()}')

In [ ]:
service = mv.create_service(
    service_name='FRAUD_DETECTION_SERVICE',
    service_compute_pool='SYSTEM_COMPUTE_POOL_CPU',
    image_build_compute_pool='SYSTEM_COMPUTE_POOL_CPU',
    ingress_enabled=True,
    max_instances=1
)

print(f'Service deployment initiated: ML.PROJECTS.FRAUD_DETECTION_SERVICE')
print(f'Compute pool: SYSTEM_COMPUTE_POOL_CPU')
print(f'Ingress enabled for REST endpoint')

In [ ]:
%%sql -r service_endpoints
SHOW ENDPOINTS IN SERVICE ML.PROJECTS.FRAUD_DETECTION_SERVICE

In [ ]:
import time

for attempt in range(30):
    status_rows = session.sql("CALL SYSTEM$GET_SERVICE_STATUS('ML.PROJECTS.FRAUD_DETECTION_SERVICE')").collect()
    status_json = status_rows[0][0]
    import json
    status = json.loads(status_json)
    svc_status = status[0].get('status', 'UNKNOWN')
    print(f'Attempt {attempt+1}: Service status = {svc_status}')
    if svc_status == 'READY':
        print('Service is READY!')
        break
    time.sleep(20)
else:
    print('Service not ready after 10 minutes. Check SHOW SERVICES.')

In [ ]:
%%sql -r endpoints_final
SHOW ENDPOINTS IN SERVICE ML.PROJECTS.FRAUD_DETECTION_SERVICE

In [ ]:
import json

print('=== Service Endpoints ===')
print(endpoints_final)
print()

endpoint_url = None
for _, row in endpoints_final.iterrows():
    if 'ingress_url' in endpoints_final.columns:
        url = row.get('ingress_url', '')
        if url:
            endpoint_url = url
            break

if endpoint_url:
    print(f'REST Endpoint URL: https://{endpoint_url}')
    print()
    print('=== Example cURL for External Inference ===')
    sample_payload = X_test.head(2).to_dict(orient='records')
    print(f"""curl -X POST \\
  "https://{endpoint_url}/predict" \\
  -H "Authorization: Snowflake Token=<your_token>" \\
  -H "Content-Type: application/json" \\
  -d '{json.dumps({"data": sample_payload}, indent=2)}'""")
else:
    print('Ingress URL not yet available. The service may still be provisioning.')
    print('Run: SHOW ENDPOINTS IN SERVICE ML.PROJECTS.FRAUD_DETECTION_SERVICE')

In [ ]:
session.sql('USE DATABASE ML').collect()
session.sql('USE SCHEMA PROJECTS').collect()

test_payload = X_test.head(3)
predictions = mv.run(test_payload, function_name='predict', service_name='FRAUD_DETECTION_SERVICE')
print('Sample predictions via SPCS service:')
print(predictions)

In [ ]:
print('='*60)
print('FRAUD DETECTION MODEL - DEPLOYMENT SUMMARY')
print('='*60)
print(f'Model:          ML.PROJECTS.FRAUD_DETECTION_XGBOOST (V1)')
print(f'Service:        ML.PROJECTS.FRAUD_DETECTION_SERVICE')
print(f'Compute Pool:   SYSTEM_COMPUTE_POOL_CPU')
print(f'Ingress:        Enabled (public REST endpoint)')
print(f'ROC-AUC:        {roc_auc:.4f}')
print(f'PR-AUC:         {pr_auc:.4f}')
print(f'Features:       {len(FEATURES)} engineered features')
print(f'Input schema:   {FEATURES}')
print('='*60)
print()
print('--- SQL Inference via SPCS Service ---')
print("SELECT ML.PROJECTS.FRAUD_DETECTION_SERVICE!predict(*)")
print("FROM (SELECT <features> FROM your_table);")
print()
print('--- Python Inference via SPCS Service ---')
print("mv.run(df, function_name='predict', service_name='FRAUD_DETECTION_SERVICE')")
print()
print('--- REST Endpoint (External) ---')
print('POST https://<ingress_url>/predict')
print('Header: Authorization: Snowflake Token=<jwt_token>')
print('Body: {"data": [{"AMOUNT": 5000, "LOG_AMOUNT": 8.52, ...}]}')

---
# REST API Load Test — 1000 Requests
Send 1000 individual inference requests to the SPCS endpoint and profile latency.

In [ ]:
import numpy as np
import pandas as pd
import json
import time
import requests
from concurrent.futures import ThreadPoolExecutor, as_completed

np.random.seed(42)
n_samples = 1000

FEATURES = [
    'AMOUNT', 'LOG_AMOUNT', 'AMOUNT_BUCKET',
    'HOUR_OF_DAY', 'IS_NIGHT', 'DAY_OF_WEEK', 'IS_WEEKEND',
    'IS_UNUSUAL_LOCATION', 'LOCATION_ENC',
    'IS_RISKY_MERCHANT', 'MERCHANT_CATEGORY_ENC',
    'IS_HIGH_AMOUNT', 'RISK_FLAG_COUNT'
]

samples = []
for i in range(n_samples):
    if i < 50:
        amt = np.random.uniform(1500, 15000)
        hour = np.random.choice([0, 1, 2, 3, 23])
        is_unusual = 1
        loc_enc = np.random.choice([0, 1, 2, 3, 4, 5, 6, 7])
        is_risky = 1
        merch_enc = np.random.choice([5, 6])
        is_high = 1
    elif i < 100:
        amt = np.random.uniform(20, 200)
        hour = np.random.randint(8, 21)
        is_unusual = 1
        loc_enc = np.random.choice([0, 1, 2, 3])
        is_risky = 0
        merch_enc = np.random.choice([0, 1, 2, 3, 4])
        is_high = 0
    else:
        amt = np.random.lognormal(mean=4.0, sigma=1.0)
        amt = min(amt, 800)
        hour = np.random.choice(range(7, 23), p=[0.02,0.05,0.08,0.10,0.12,0.12,0.10,0.10,0.08,0.07,0.06,0.04,0.03,0.02,0.005,0.005])
        is_unusual = 0
        loc_enc = np.random.randint(8, 23)
        is_risky = int(np.random.random() < 0.02)
        merch_enc = np.random.choice([0, 1, 2, 3, 4])
        is_high = int(amt > 1000)

    log_amt = float(np.log1p(amt))
    if amt < 50: bucket = 0
    elif amt < 200: bucket = 1
    elif amt < 500: bucket = 2
    elif amt < 1000: bucket = 3
    elif amt < 2500: bucket = 4
    elif amt < 5000: bucket = 5
    else: bucket = 6

    dow = np.random.randint(0, 7)
    is_weekend = int(dow >= 5)
    is_night = int(hour < 7 or hour > 22)
    risk_count = is_night + is_unusual + is_risky + is_high

    samples.append({
        'AMOUNT': round(float(amt), 2),
        'LOG_AMOUNT': round(log_amt, 6),
        'AMOUNT_BUCKET': bucket,
        'HOUR_OF_DAY': int(hour),
        'IS_NIGHT': is_night,
        'DAY_OF_WEEK': dow,
        'IS_WEEKEND': is_weekend,
        'IS_UNUSUAL_LOCATION': is_unusual,
        'LOCATION_ENC': int(loc_enc),
        'IS_RISKY_MERCHANT': is_risky,
        'MERCHANT_CATEGORY_ENC': int(merch_enc),
        'IS_HIGH_AMOUNT': is_high,
        'RISK_FLAG_COUNT': risk_count
    })

print(f'Generated {len(samples)} sample requests')
print(f'  Suspicious (high-risk fraud-like): 50')
print(f'  Ambiguous (unusual location, normal amount): 50')
print(f'  Legitimate patterns: 900')

In [ ]:
import time
import requests
import json

ENDPOINT_URL = 'https://mqlvigb-kqbczbp-qub39712.snowflakecomputing.app'
INFERENCE_PATH = '/predict'

token = session.connection._rest._token
headers = {
    'Authorization': f'Snowflake Token="{token}"',
    'Content-Type': 'application/json'
}

def send_single_rest(sample):
    payload = {'data': [sample]}
    start = time.perf_counter()
    try:
        resp = requests.post(
            f'{ENDPOINT_URL}{INFERENCE_PATH}',
            json=payload,
            headers=headers,
            timeout=30
        )
        elapsed_ms = (time.perf_counter() - start) * 1000
        return {'latency_ms': elapsed_ms, 'status': resp.status_code, 'prediction': resp.json() if resp.status_code == 200 else None, 'error': None if resp.status_code == 200 else resp.text[:200]}
    except Exception as e:
        elapsed_ms = (time.perf_counter() - start) * 1000
        return {'latency_ms': elapsed_ms, 'status': -1, 'prediction': None, 'error': str(e)[:200]}

def send_via_registry(batch_df):
    start = time.perf_counter()
    try:
        result = mv.run(batch_df, function_name='predict', service_name='FRAUD_DETECTION_SERVICE')
        elapsed_ms = (time.perf_counter() - start) * 1000
        return elapsed_ms, result, None
    except Exception as e:
        elapsed_ms = (time.perf_counter() - start) * 1000
        return elapsed_ms, None, str(e)

rest_test = send_single_rest(samples[0])
print(f"REST API test: status={rest_test['status']}, latency={rest_test['latency_ms']:.1f}ms")
if rest_test['status'] != 200:
    print(f"REST not reachable from notebook ({rest_test['error'][:80]}...)")
    print('Using mv.run() via SPCS service for latency profiling instead.')
    USE_REST = False
else:
    print('REST endpoint reachable. Using direct HTTP calls.')
    USE_REST = True

In [ ]:
all_samples_df = pd.DataFrame(samples)

batch_sizes = [1, 5, 10, 25, 50, 100]
batch_results = {}

for bs in batch_sizes:
    latencies_list = []
    n_batches = min(1000 // bs, 200)
    for i in range(n_batches):
        batch = all_samples_df.iloc[i*bs:(i+1)*bs].reset_index(drop=True)
        start = time.perf_counter()
        _ = mv.run(batch, function_name='predict', service_name='FRAUD_DETECTION_SERVICE')
        elapsed_ms = (time.perf_counter() - start) * 1000
        latencies_list.append(elapsed_ms)
    batch_results[bs] = latencies_list
    total_inferences = n_batches * bs
    avg_lat = np.mean(latencies_list)
    p95_lat = np.percentile(latencies_list, 95)
    throughput = total_inferences / (sum(latencies_list) / 1000)
    print(f'Batch={bs:>3d} | Requests={n_batches:>4d} | Inferences={total_inferences:>5d} | '
          f'Avg={avg_lat:>7.1f}ms | P95={p95_lat:>7.1f}ms | Throughput={throughput:>6.1f} inf/s')

single_latencies = batch_results[1]
results_df = pd.DataFrame({
    'idx': range(len(single_latencies)),
    'latency_ms': single_latencies,
    'status': [200] * len(single_latencies)
})
latencies = pd.Series(single_latencies)
overall_elapsed = sum(single_latencies) / 1000
print(f'\nSingle-request profiling: {len(single_latencies)} requests in {overall_elapsed:.1f}s')

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

fig, axes = plt.subplots(2, 2, figsize=(16, 10))

axes[0,0].hist(latencies, bins=50, color='steelblue', edgecolor='white', alpha=0.85)
axes[0,0].axvline(latencies.median(), color='red', linestyle='--', label=f'Median: {latencies.median():.0f}ms')
axes[0,0].axvline(latencies.quantile(0.95), color='orange', linestyle='--', label=f'P95: {latencies.quantile(0.95):.0f}ms')
axes[0,0].set_title('Single-Request Latency Distribution')
axes[0,0].set_xlabel('Latency (ms)')
axes[0,0].set_ylabel('Count')
axes[0,0].legend()

axes[0,1].plot(range(len(single_latencies)), single_latencies,
               linewidth=0.5, alpha=0.7, color='steelblue')
axes[0,1].axhline(latencies.median(), color='red', linestyle='--', alpha=0.7, label='Median')
axes[0,1].axhline(latencies.quantile(0.95), color='orange', linestyle='--', alpha=0.7, label='P95')
axes[0,1].set_title('Latency Over Request Sequence')
axes[0,1].set_xlabel('Request #')
axes[0,1].set_ylabel('Latency (ms)')
axes[0,1].legend()

batch_sizes_plot = sorted(batch_results.keys())
avg_lats = [np.mean(batch_results[bs]) for bs in batch_sizes_plot]
p95_lats = [np.percentile(batch_results[bs], 95) for bs in batch_sizes_plot]
throughputs = [(min(1000//bs,200)*bs)/(sum(batch_results[bs])/1000) for bs in batch_sizes_plot]

ax3 = axes[1,0]
color1 = 'steelblue'
ax3.bar([str(bs) for bs in batch_sizes_plot], avg_lats, color=color1, alpha=0.7, label='Avg Latency')
ax3.bar([str(bs) for bs in batch_sizes_plot], p95_lats, color='orange', alpha=0.3, label='P95 Latency')
ax3.set_title('Latency by Batch Size')
ax3.set_xlabel('Batch Size')
ax3.set_ylabel('Latency (ms)')
ax3.legend()

ax4 = axes[1,1]
ax4.plot([str(bs) for bs in batch_sizes_plot], throughputs, 'o-', color='#2ecc71', linewidth=2, markersize=8)
for i, (bs, tp) in enumerate(zip(batch_sizes_plot, throughputs)):
    ax4.annotate(f'{tp:.0f}', (str(bs), tp), textcoords='offset points', xytext=(0,10), ha='center', fontweight='bold')
ax4.set_title('Throughput by Batch Size')
ax4.set_xlabel('Batch Size')
ax4.set_ylabel('Inferences/sec')
ax4.grid(axis='y', alpha=0.3)

plt.suptitle('SPCS Inference Latency Profile — Fraud Detection Model', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()